# LV 3D Shape Reconstruction — Implicit Neural Representation (Occupancy Network)

## Approach 3: INR / Occupancy Network

**Core idea:** Instead of deforming a template mesh, we train a neural network to represent the
LV surface as a *continuous occupancy function*:

```
f(z_latent, x, y, z_query) → P(inside_endo), P(inside_epi)  ∈ [0, 1]²
```

The 3D surface is the **zero-level set** (iso-surface at threshold 0.5), extracted at inference
via Marching Cubes on a dense query grid.

### Pipeline
```
SAX contour points (xyz + tissue label)
        │
   PointNet Encoder  →  latent code z  (256-dim)
        │
   INR Decoder  ←─── Fourier positional encoding of query point (x,y,z)
        │
   [endo_occ, epi_occ]  (per query point)
        │
   Marching Cubes at 0.5 threshold  →  two closed 3D meshes
        │
   Wall thickness  =  epi_mesh  −  endo_mesh
```

### Why this solves the previous problems
- **No SSM needed**: purely data-driven, no template topology
- **Clean endo/epi separation**: two independent output channels, each supervised by
  ground-truth occupancy from the corresponding surface
- **Arbitrary resolution**: query any grid resolution at inference
- **Uncertainty**: sample the latent space → ensemble of plausible shapes


## 1. Setup

In [9]:
%pip install -q vtk scikit-image tqdm scipy nibabel scikit-learn trimesh

Note: you may need to restart the kernel to use updated packages.


In [10]:
import os, gzip, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from pathlib import Path
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
import vtk
from skimage.measure import marching_cubes
import trimesh

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f'  {p.name}  |  {p.total_memory/1024**3:.1f} GB')


Device: cuda
  Tesla T4  |  14.6 GB


## 2. Configuration

In [11]:
CFG = dict(
    # ── Paths ──────────────────────────────────────────────────────────
    mesh_dir     = '/kaggle/input/datasets/andrefce/meshes/generated_meshes',
    ssm_dir      = 'Statistical-Shape-Model',
    output_dir   = '/kaggle/working',
    ckpt_path    = 'checkpoints/inr_lv.pth',

    # ── SAX contour extraction ──────────────────────────────────────────
    n_slices     = 10,
    slice_thick  = 8.0,
    slice_space  = 10.0,
    pts_per_ring = 40,       # points per endo/epi ring per slice

    # ── Occupancy sampling ──────────────────────────────────────────────
    n_query      = 2048,     # query points per sample during training
    surface_std  = 2.0,      # mm — Gaussian noise on surface points
    epi_offset   = 8.0,      # mm — outward normal offset for epi surface

    # ── Architecture ────────────────────────────────────────────────────
    latent_dim   = 256,
    fourier_L    = 6,        # Fourier positional encoding levels
    decoder_hidden = 512,
    decoder_layers = 8,
    skip_layer   = 4,        # layer index that receives skip from input

    # ── Training ────────────────────────────────────────────────────────
    epochs       = 400,
    batch_size   = 16,
    lr           = 1e-4,
    weight_decay = 1e-5,
    patience     = 80,

    # ── Inference ───────────────────────────────────────────────────────
    grid_res     = 64,       # voxel grid resolution for marching cubes
    iso_thresh   = 0.5,

    seed         = 42,
)

os.makedirs(os.path.dirname(CFG['ckpt_path']), exist_ok=True)
os.makedirs(CFG['output_dir'], exist_ok=True)
print('✅ Config ready')


✅ Config ready


## 3. Mesh Loading + SSM

In [12]:
def load_vtk_mesh(path):
    """Load VTK polydata → vertices + triangulated faces."""
    reader = vtk.vtkPolyDataReader()
    reader.SetFileName(str(path))
    reader.Update()
    poly = reader.GetOutput()

    # Force triangulation
    tri = vtk.vtkTriangleFilter()
    tri.SetInputData(poly)
    tri.Update()
    poly = tri.GetOutput()

    pts = poly.GetPoints()
    verts = np.array([pts.GetPoint(i) for i in range(pts.GetNumberOfPoints())],
                     dtype=np.float32)

    cells_vtk = poly.GetPolys()
    cells_vtk.InitTraversal()
    id_list = vtk.vtkIdList()
    faces = []
    while cells_vtk.GetNextCell(id_list):
        if id_list.GetNumberOfIds() == 3:
            faces.append([id_list.GetId(j) for j in range(3)])

    return verts, np.array(faces, dtype=np.int64)


def load_vtk_points(path):
    reader = vtk.vtkPolyDataReader()
    reader.SetFileName(str(path))
    reader.Update()
    poly = reader.GetOutput()
    pts  = poly.GetPoints()
    return np.array([pts.GetPoint(i) for i in range(pts.GetNumberOfPoints())],
                    dtype=np.float32)


def load_csv_gz(path):
    with gzip.open(str(path), 'rt') as f:
        return np.loadtxt(f, delimiter=',').astype(np.float32)


# ── Load SSM endo surface ────────────────────────────────────────────────────
SSM_DIR = CFG['ssm_dir']
if not os.path.exists(SSM_DIR):
    os.system('git clone --quiet https://github.com/UK-Digital-Heart-Project/Statistical-Shape-Model.git')
    print('✅ SSM cloned')

ENDO_VERTS, ENDO_FACES = load_vtk_mesh(f'{SSM_DIR}/LV_ED_mean.vtk')
PCS     = load_csv_gz(f'{SSM_DIR}/LV_ED_pc_100_modes.csv.gz')
LAMBDAS = load_csv_gz(f'{SSM_DIR}/LV_ED_var_100_modes.csv.gz').ravel()
SIGMAS  = np.sqrt(LAMBDAS)
K_MODES = 40
SCALED_PCS = PCS[:, :K_MODES] * SIGMAS[:K_MODES]

print(f'SSM endo mesh  : {ENDO_VERTS.shape[0]} vertices | {len(ENDO_FACES)} faces')
assert len(ENDO_FACES) > 0, (
    'No faces loaded! VTK file may not contain triangles — check vtkTriangleFilter.')

# ── Dataset metadata ─────────────────────────────────────────────────────────
with open(f'{CFG["mesh_dir"]}/metadata.json') as f:
    raw_meta = json.load(f)
metadata = [m for m in raw_meta if m.get('accepted', True)]
B_full   = np.array([m['b'] for m in metadata], dtype=np.float32)
print(f'Accepted meshes: {len(metadata)}')


SSM endo mesh  : 22043 vertices | 43840 faces
Accepted meshes: 1000


In [13]:
# --- Define the missing split logic ---
from sklearn.model_selection import train_test_split

# Create a list of all indices
all_indices = np.arange(len(metadata))

# 1. Split into (Train+Val) and Test (e.g., 90% / 10%)
tr_val_idx, te_idx = train_test_split(all_indices, test_size=0.1, random_state=CFG['seed'])

# 2. Split (Train+Val) into Train and Val (e.g., 80% / 20% of the remainder)
tr_idx, val_idx = train_test_split(tr_val_idx, test_size=0.2, random_state=CFG['seed'])

# Create the corresponding metadata lists
tr_meta  = [metadata[i] for i in tr_idx]
val_meta = [metadata[i] for i in val_idx]
te_meta  = [metadata[i] for i in te_idx]

print(f"Splits ready: Train={len(tr_idx)}, Val={len(val_idx)}, Test={len(te_idx)}")

Splits ready: Train=720, Val=180, Test=100


## 4. Generate Epi Surface via Outward Normal Offset

In [14]:
"""
The UK Digital Heart Project SSM provides the endocardial surface only.
We approximate the epicardial surface by displacing endo vertices outward
along their surface normals by `epi_offset` mm (default 8 mm — typical LV wall
thickness at ED).  This is a valid anatomical approximation; a real study would
use a paired epi SSM (e.g. from ACDC or UK Biobank segmentations).
"""

def compute_vertex_normals_np(verts, faces):
    """Per-vertex unit normals (NumPy, used for GT generation only)."""
    v0, v1, v2 = verts[faces[:,0]], verts[faces[:,1]], verts[faces[:,2]]
    fn = np.cross(v1 - v0, v2 - v0)                      # (F, 3) face normals
    vn = np.zeros_like(verts)
    for i in range(3):
        np.add.at(vn, faces[:,i], fn)
    norms = np.linalg.norm(vn, axis=1, keepdims=True).clip(min=1e-8)
    return (vn / norms).astype(np.float32)


def generate_epi_from_endo(endo_verts, endo_faces, offset_mm=8.0):
    """
    Create an approximate epicardial surface by offsetting endo along normals.
    Returns (epi_verts, epi_faces) — same topology as endo.
    """
    normals   = compute_vertex_normals_np(endo_verts, endo_faces)
    epi_verts = endo_verts + normals * offset_mm
    # Flip face winding so outer surface normals point outward
    epi_faces = endo_faces[:, [0, 2, 1]]
    return epi_verts.astype(np.float32), epi_faces


def sample_occupancy_gt(endo_verts, endo_faces, epi_verts, epi_faces,
                         n_query=2048, surface_std=2.0, bbox_pad=15.0):
    """
    Sample query points and compute binary occupancy labels for endo and epi.

    Sampling strategy (balanced):
      - 40%  near endo surface  (surface pts + Gaussian noise)
      - 40%  near epi  surface  (surface pts + Gaussian noise)
      - 20%  uniform in bounding box

    Labels:
      endo_occ[i] = 1  if query_point[i] is INSIDE the endo surface
      epi_occ[i]  = 1  if query_point[i] is INSIDE the epi surface

    Uses trimesh.contains() which runs a fast ray-casting test.
    """
    rng = np.random.default_rng()

    # Near-surface samples
    n_surf = int(n_query * 0.4)
    n_rand = n_query - 2 * n_surf

    # Endo surface noise
    e_idx  = rng.integers(0, len(endo_verts), n_surf)
    e_pts  = endo_verts[e_idx] + rng.normal(0, surface_std, (n_surf, 3))

    # Epi surface noise
    p_idx  = rng.integers(0, len(epi_verts), n_surf)
    p_pts  = epi_verts[p_idx] + rng.normal(0, surface_std, (n_surf, 3))

    # Uniform random in bounding box
    all_pts  = np.vstack([endo_verts, epi_verts])
    lo, hi   = all_pts.min(0) - bbox_pad, all_pts.max(0) + bbox_pad
    rand_pts = rng.uniform(lo, hi, (n_rand, 3)).astype(np.float32)

    query = np.vstack([e_pts, p_pts, rand_pts]).astype(np.float32)

    # Occupancy labels via ray-casting (trimesh)
    endo_mesh = trimesh.Trimesh(vertices=endo_verts, faces=endo_faces, process=False)
    epi_mesh  = trimesh.Trimesh(vertices=epi_verts,  faces=epi_faces,  process=False)

    endo_occ = endo_mesh.contains(query).astype(np.float32)
    epi_occ  = epi_mesh.contains(query).astype(np.float32)

    return query, endo_occ, epi_occ


# ── Quick sanity check on first mesh ─────────────────────────────────────────
_m      = metadata[0]
_idx    = _m.get('saved_idx', 0)
_e_v, _e_f = load_vtk_mesh(f'{CFG["mesh_dir"]}/mesh_{_idx:04d}.vtk')
if len(_e_f) == 0:
    print('⚠  No faces in generated mesh — using SSM mean endo for demo')
    _e_v, _e_f = ENDO_VERTS.copy(), ENDO_FACES.copy()

_p_v, _p_f = generate_epi_from_endo(_e_v, _e_f, CFG['epi_offset'])
_q, _eo, _po = sample_occupancy_gt(_e_v, _e_f, _p_v, _p_f,
                                    CFG['n_query'], CFG['surface_std'])

print(f'Epi surface     : {_p_v.shape[0]} verts | {len(_p_f)} faces')
print(f'Query points    : {_q.shape}')
print(f'Endo occ balance: {_eo.mean()*100:.1f}% inside')
print(f'Epi  occ balance: {_po.mean()*100:.1f}% inside')


Epi surface     : 22043 verts | 43840 faces
Query points    : (2048, 3)
Endo occ balance: 29.8% inside
Epi  occ balance: 43.8% inside


## 5. SAX Contour Extraction (Encoder Input)

In [15]:
def robust_tissue_labels(radii):
    """K-means(2) on radii → endo (0) / epi (1). Label 0 = smaller mean radius."""
    n = len(radii)
    if n < 6:
        return (radii >= np.median(radii)).astype(np.float32)
    try:
        km = KMeans(n_clusters=2, n_init=5, random_state=42, max_iter=100)
        labels = km.fit_predict(radii.reshape(-1, 1))
        if km.cluster_centers_[0, 0] > km.cluster_centers_[1, 0]:
            labels = 1 - labels
        return labels.astype(np.float32)
    except Exception:
        return (radii >= np.median(radii)).astype(np.float32)


def angular_order(xy):
    """Return indices of xy sorted by angle around centroid."""
    c = xy.mean(0)
    return np.argsort(np.arctan2(xy[:, 1] - c[1], xy[:, 0] - c[0]))


def extract_sax_contours(points, n_slices=10, thick=8.0, space=10.0,
                          pts_per_ring=40):
    """
    Extract endo + epi contour rings from a 3D mesh point cloud.

    Returns
    -------
    xyz    : (M, 3)  contour point positions
    tissue : (M,)    0 = endo,  1 = epi
    """
    z_min, z_max = points[:, 2].min(), points[:, 2].max()
    start   = z_min + (z_max - z_min - space * (n_slices - 1)) / 2.0
    z_ctrs  = start + np.arange(n_slices) * space
    half    = thick / 2.0

    all_xyz, all_tissue = [], []

    for zc in z_ctrs:
        mask = np.abs(points[:, 2] - zc) <= half
        if mask.sum() < 8:
            continue
        xy = points[mask, :2]
        c  = xy.mean(0)
        r  = np.linalg.norm(xy - c, axis=1).astype(np.float32)
        t  = robust_tissue_labels(r)

        for label in [0.0, 1.0]:
            ring_xy = xy[t == label]
            if len(ring_xy) < 3:
                continue
            ord_idx = angular_order(ring_xy)
            # Subsample to pts_per_ring
            step = max(1, len(ring_xy) // pts_per_ring)
            sel  = ord_idx[::step][:pts_per_ring]
            ring = ring_xy[sel]
            z_col = np.full(len(ring), zc, dtype=np.float32)
            all_xyz.append(np.column_stack([ring, z_col]))
            all_tissue.append(np.full(len(ring), label, dtype=np.float32))

    if not all_xyz:
        return np.empty((0, 3), np.float32), np.empty(0, np.float32)

    return np.vstack(all_xyz).astype(np.float32), np.concatenate(all_tissue)


# ── Test ─────────────────────────────────────────────────────────────────────
_pts = load_vtk_points(f'{CFG["mesh_dir"]}/mesh_{_idx:04d}.vtk')
_xyz, _t = extract_sax_contours(_pts, CFG['n_slices'], CFG['slice_thick'],
                                  CFG['slice_space'], CFG['pts_per_ring'])
print(f'SAX contour cloud : {_xyz.shape[0]} points')
print(f'Endo fraction     : {(_t==0).mean()*100:.1f}%  |  Epi: {(_t==1).mean()*100:.1f}%')


SAX contour cloud : 800 points
Endo fraction     : 50.0%  |  Epi: 50.0%


## 6. Dataset

In [ ]:
# ============================================================
# FAST OCCUPANCY CACHE BUILDER (PARALLEL + PRACTICAL SPEED KNOBS)
# ============================================================

import os
import time
import numpy as np
import trimesh
from tqdm.notebook import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp

CACHE_DIR = "./occupancy_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# ---- Speed knobs (safe defaults) ----
CACHE_N_QUERY = int(CFG.get('cache_n_query', min(CFG['n_query'], 512)))  # was 2048
BUILD_TEST_CACHE = bool(CFG.get('build_test_cache', False))               # test cache can wait
N_WORKERS = int(CFG.get('cache_workers', max(1, min(8, (mp.cpu_count() or 2) - 1))))

print(f"Cache settings → n_query={CACHE_N_QUERY}, workers={N_WORKERS}, build_test={BUILD_TEST_CACHE}")


def _mesh_path_from_row(row, mesh_dir):
    """Resolve mesh path robustly from metadata row."""
    if isinstance(row, dict):
        if 'saved_idx' in row:
            return os.path.join(mesh_dir, f"mesh_{int(row['saved_idx']):04d}.vtk")
        for k in ('mesh_name', 'mesh_file', 'filename', 'file', 'path'):
            if k in row:
                v = str(row[k])
                return v if os.path.isabs(v) else os.path.join(mesh_dir, v)

    if hasattr(row, 'to_dict'):
        d = row.to_dict()
        if 'saved_idx' in d:
            return os.path.join(mesh_dir, f"mesh_{int(d['saved_idx']):04d}.vtk")
        for k in ('mesh_name', 'mesh_file', 'filename', 'file', 'path'):
            if k in d:
                v = str(d[k])
                return v if os.path.isabs(v) else os.path.join(mesh_dir, v)

    v = str(row)
    return v if os.path.isabs(v) else os.path.join(mesh_dir, v)


def _sample_query_points(endo_v, epi_v, n_query, surface_std=2.0, bbox_pad=12.0):
    """Balanced query sampling: near endo + near epi + random bbox."""
    rng = np.random.default_rng()
    n_surf = int(n_query * 0.4)
    n_rand = n_query - 2 * n_surf

    e_idx = rng.integers(0, len(endo_v), n_surf)
    p_idx = rng.integers(0, len(epi_v),  n_surf)

    e_pts = endo_v[e_idx] + rng.normal(0, surface_std, (n_surf, 3))
    p_pts = epi_v[p_idx]  + rng.normal(0, surface_std, (n_surf, 3))

    all_pts = np.vstack([endo_v, epi_v])
    lo, hi = all_pts.min(0) - bbox_pad, all_pts.max(0) + bbox_pad
    r_pts = rng.uniform(lo, hi, (n_rand, 3))

    return np.vstack([e_pts, p_pts, r_pts]).astype(np.float32)


def _build_one_cache(task):
    global_idx, row, mesh_dir, cfg, cache_dir, cache_n_query = task
    out_path = os.path.join(cache_dir, f"{global_idx}.npz")
    if os.path.exists(out_path):
        return global_idx, 'skip', None

    try:
        mesh_path = _mesh_path_from_row(row, mesh_dir)
        endo_v, endo_f = load_vtk_mesh(mesh_path)
        if len(endo_f) == 0:
            return global_idx, 'empty_faces', mesh_path

        epi_v, epi_f = generate_epi_from_endo(
            endo_v, endo_f, offset_mm=cfg.get('epi_offset', 8.0)
        )

        query = _sample_query_points(
            endo_v, epi_v,
            n_query=cache_n_query,
            surface_std=cfg.get('surface_std', 2.0),
            bbox_pad=12.0,
        )

        # NOTE: np.savez (uncompressed) is much faster than np.savez_compressed.
        endo_mesh = trimesh.Trimesh(vertices=endo_v, faces=endo_f, process=False)
        epi_mesh  = trimesh.Trimesh(vertices=epi_v,  faces=epi_f,  process=False)

        endo_occ = endo_mesh.contains(query).astype(np.float32)
        epi_occ  = epi_mesh.contains(query).astype(np.float32)

        np.savez(
            out_path,
            query=query.astype(np.float32),
            endo_occ=endo_occ,
            epi_occ=epi_occ,
        )
        return global_idx, 'ok', None
    except Exception as e:
        return global_idx, 'error', str(e)


def _iter_tasks(meta_subset, global_indices, mesh_dir, cfg, cache_n_query):
    for global_idx, row in zip(global_indices, meta_subset):
        yield (int(global_idx), row, mesh_dir, cfg, CACHE_DIR, cache_n_query)


def build_cache_parallel(meta_subset, global_indices, mesh_dir, cfg, desc='Split', n_workers=4, cache_n_query=512):
    tasks = list(_iter_tasks(meta_subset, global_indices, mesh_dir, cfg, cache_n_query))
    if len(tasks) == 0:
        print(f'⚠ {desc}: no tasks')
        return

    t0 = time.time()
    done_ok = done_skip = done_err = 0
    examples = []

    print(f'⚡ Building {desc} cache with {n_workers} workers...')
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        futs = [ex.submit(_build_one_cache, t) for t in tasks]
        for fut in tqdm(as_completed(futs), total=len(futs), desc=desc):
            idx, status, msg = fut.result()
            if status == 'ok':
                done_ok += 1
            elif status == 'skip':
                done_skip += 1
            else:
                done_err += 1
                if len(examples) < 5:
                    examples.append((idx, status, msg))

    dt = time.time() - t0
    print(f'✅ {desc}: ok={done_ok}, skipped={done_skip}, errors={done_err} | {dt/60:.1f} min')
    if examples:
        print('   Sample issues:')
        for idx, st, msg in examples:
            print(f'   - idx={idx} [{st}] {msg}')


print('🚀 Starting optimized occupancy cache build...')
build_cache_parallel(tr_meta, tr_idx, CFG['mesh_dir'], CFG, desc='Train', n_workers=N_WORKERS, cache_n_query=CACHE_N_QUERY)
build_cache_parallel(val_meta, val_idx, CFG['mesh_dir'], CFG, desc='Val',   n_workers=N_WORKERS, cache_n_query=CACHE_N_QUERY)
if BUILD_TEST_CACHE:
    build_cache_parallel(te_meta, te_idx, CFG['mesh_dir'], CFG, desc='Test', n_workers=N_WORKERS, cache_n_query=CACHE_N_QUERY)
else:
    print('⏭ Skipping Test cache for now (set CFG[\'build_test_cache\']=True to enable).')
print('✅ Occupancy cache build complete.')

Using device: cuda
🚀 Starting FAST GPU occupancy build...
🔥 Building Train cache on GPU...


Train:   0%|          | 0/720 [00:00<?, ?it/s]


KeyError: 'mesh_name'

## 7. Architecture

In [ ]:
"""
### PointNet Encoder
Processes the variable-size SAX contour cloud → fixed-size latent code.

Input: (B, N, 4)  — [x, y, z, tissue_label]
Mask:  (B, N)     — True for real points, False for padding

Key design:
- Shared MLP lifts each point: 4 → 64 → 128 → 256
- Global max-pool over all real points → g_all (256)
- Per-tissue max-pool (endo and epi separately) → g_endo, g_epi (128 each)
- Concatenate [g_all, g_endo, g_epi] → 512 → Linear → latent_dim (256)

Separating endo/epi at the pooling stage forces the encoder to maintain
distinct representations for each tissue type.
"""

class PointNetEncoder(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Linear(4, 64),  nn.LayerNorm(64),  nn.ReLU())
        self.conv2 = nn.Sequential(
            nn.Linear(64, 128), nn.LayerNorm(128), nn.ReLU())
        self.conv3 = nn.Sequential(
            nn.Linear(128, 256), nn.LayerNorm(256), nn.ReLU())

        # Per-tissue projection (endo + epi separately)
        self.tissue_proj = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU())

        # Final aggregation: global(256) + endo(128) + epi(128) = 512
        self.agg = nn.Sequential(
            nn.Linear(512, 512), nn.LayerNorm(512), nn.ReLU(),
            nn.Linear(512, latent_dim))

    def forward(self, x, mask):
        """
        x    : (B, N, 4)
        mask : (B, N) bool  — True = real point
        Returns z : (B, latent_dim)
        """
        B, N, _ = x.shape
        h = self.conv1(x)   # (B, N, 64)
        h = self.conv2(h)   # (B, N, 128)
        h = self.conv3(h)   # (B, N, 256)

        # Global max-pool (masked)
        neg_inf = torch.full_like(h, -1e9)
        h_masked = torch.where(mask.unsqueeze(-1), h, neg_inf)
        g_all = h_masked.max(dim=1).values   # (B, 256)

        # Per-tissue max-pool
        tissue = x[:, :, 3]                  # (B, N)  0=endo, 1=epi
        hp = self.tissue_proj(h)             # (B, N, 128)

        def tissue_pool(label):
            t_mask = (tissue == label) & mask
            t_neg  = torch.full_like(hp, -1e9)
            t_h    = torch.where(t_mask.unsqueeze(-1), hp, t_neg)
            pooled = t_h.max(dim=1).values   # (B, 128)
            # If no points of this tissue type, fall back to zeros
            has_pts = t_mask.any(dim=1, keepdim=True).float()
            return pooled * has_pts

        g_endo = tissue_pool(0.0)   # (B, 128)
        g_epi  = tissue_pool(1.0)   # (B, 128)

        cat = torch.cat([g_all, g_endo, g_epi], dim=-1)   # (B, 512)
        return self.agg(cat)                               # (B, latent_dim)


In [ ]:
"""
### Fourier Positional Encoding
Maps 3D query coordinates → higher-frequency features.
pe(xyz) = [xyz, sin(2^0 π xyz), cos(2^0 π xyz), ..., sin(2^{L-1} π xyz), cos(2^{L-1} π xyz)]
Output dim = 3 + 6*L
"""

class FourierPE(nn.Module):
    def __init__(self, L=6):
        super().__init__()
        self.L    = L
        # register_buffer so freqs move with .to(device) automatically
        self.register_buffer('freqs', 2.0 ** torch.arange(L).float() * np.pi)
        self.out_dim = 3 + 6 * L

    def forward(self, xyz):
        """xyz : (..., 3)  →  (..., 3 + 6L)"""
        # freqs is already on the correct device via register_buffer
        angles = xyz.unsqueeze(-1) * self.freqs       # (..., 3, L)
        enc    = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)  # (..., 3, 2L)
        enc    = enc.reshape(*xyz.shape[:-1], 6 * self.L)
        return torch.cat([xyz, enc], dim=-1)           # (..., 3 + 6L)


In [ ]:
"""
### INR Decoder
8-layer MLP with skip connection at layer 4.
Input: [z (latent_dim), pe(xyz) (3 + 6L)]
Output: [endo_logit, epi_logit]
"""

class INRDecoder(nn.Module):
    def __init__(self, latent_dim=256, fourier_L=6,
                 hidden=512, n_layers=8, skip_layer=4):
        super().__init__()
        self.skip_layer = skip_layer
        pe_dim = 3 + 6 * fourier_L
        in_dim = latent_dim + pe_dim

        layers = []
        cur_dim = in_dim
        for i in range(n_layers):
            # Skip connection: re-inject input at skip_layer
            if i == skip_layer:
                cur_dim += in_dim
            layers.append(nn.Linear(cur_dim, hidden))
            layers.append(nn.LayerNorm(hidden))
            layers.append(nn.ReLU())
            cur_dim = hidden
        self.layers = nn.ModuleList(layers)

        self.head_endo = nn.Linear(hidden, 1)
        self.head_epi  = nn.Linear(hidden, 1)

        self._in_dim = in_dim
        self._chunk_size = 3   # 3 = Linear + LayerNorm + ReLU

        # Initialise heads near zero so predictions start near 0.5
        nn.init.normal_(self.head_endo.weight, std=1e-4)
        nn.init.zeros_(self.head_endo.bias)
        nn.init.normal_(self.head_epi.weight,  std=1e-4)
        nn.init.zeros_(self.head_epi.bias)

    def forward(self, z, pe_xyz):
        """
        z      : (B, latent_dim)
        pe_xyz : (B, Q, pe_dim)
        Returns:
          endo_logit : (B, Q)
          epi_logit  : (B, Q)
        """
        B, Q, _ = pe_xyz.shape
        z_exp = z.unsqueeze(1).expand(-1, Q, -1)      # (B, Q, latent_dim)
        inp   = torch.cat([z_exp, pe_xyz], dim=-1)     # (B, Q, in_dim)

        h    = inp
        step = 0   # which layer index (not module index)
        for j in range(0, len(self.layers), 3):        # step through Linear+LN+ReLU triplets
            if step == self.skip_layer:
                h = torch.cat([h, inp], dim=-1)
            h    = self.layers[j](h)                   # Linear
            h    = self.layers[j+1](h)                 # LayerNorm
            h    = self.layers[j+2](h)                 # ReLU
            step += 1

        endo_logit = self.head_endo(h).squeeze(-1)     # (B, Q)
        epi_logit  = self.head_epi(h).squeeze(-1)      # (B, Q)
        return endo_logit, epi_logit


In [ ]:
class OccupancyNetwork(nn.Module):
    """Full occupancy network = PointNet encoder + Fourier PE + INR decoder."""

    def __init__(self, cfg):
        super().__init__()
        self.encoder  = PointNetEncoder(latent_dim=cfg['latent_dim'])
        self.fourier  = FourierPE(L=cfg['fourier_L'])
        self.decoder  = INRDecoder(
            latent_dim    = cfg['latent_dim'],
            fourier_L     = cfg['fourier_L'],
            hidden        = cfg['decoder_hidden'],
            n_layers      = cfg['decoder_layers'],
            skip_layer    = cfg['skip_layer'],
        )

    def encode(self, contour, mask):
        """contour: (B,N,4), mask: (B,N) → z: (B, latent_dim)"""
        return self.encoder(contour, mask)

    def decode(self, z, query_xyz):
        """z: (B, D), query_xyz: (B, Q, 3) → endo_logit, epi_logit: (B, Q)"""
        pe = self.fourier(query_xyz)         # (B, Q, pe_dim)
        return self.decoder(z, pe)

    def forward(self, contour, mask, query_xyz):
        z = self.encode(contour, mask)
        return self.decode(z, query_xyz)

    def num_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


model = OccupancyNetwork(CFG).to(DEVICE)
print(f'Model parameters: {model.num_params():,}')
print(f'  Encoder : {model.encoder.num_parameters() if hasattr(model.encoder, "num_parameters") else sum(p.numel() for p in model.encoder.parameters()):,}')
print(f'  Decoder : {sum(p.numel() for p in model.decoder.parameters()):,}')


## 8. Training

In [ ]:
def occupancy_loss(endo_logit, epi_logit, endo_occ, epi_occ):
    """
    Binary cross-entropy for each channel.
    endo_occ and epi_occ must be in [0, 1].

    We use a boundary-aware weighting: points near the surface boundary
    (where occupancy switches from 0→1) are upweighted 3× to sharpen the
    predicted iso-surface.  Boundary points are those with GT in (0.1, 0.9).
    """
    bce_endo = F.binary_cross_entropy_with_logits(endo_logit, endo_occ, reduction='none')
    bce_epi  = F.binary_cross_entropy_with_logits(epi_logit,  epi_occ,  reduction='none')

    # Boundary weight: near-boundary points (GT ≈ 0.5 from noisy surface sampling)
    # are naturally those with |GT - 0.5| < 0.4 — weight them more
    w_endo = 1.0 + 2.0 * (endo_occ - 0.5).abs().lt(0.4).float()
    w_epi  = 1.0 + 2.0 * (epi_occ  - 0.5).abs().lt(0.4).float()

    loss = (bce_endo * w_endo).mean() + (bce_epi * w_epi).mean()
    return loss, bce_endo.mean().item(), bce_epi.mean().item()


optimizer = torch.optim.AdamW(model.parameters(),
                               lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['epochs'], eta_min=CFG['lr'] * 0.01)
use_amp   = (DEVICE.type == 'cuda')
scaler    = torch.amp.GradScaler('cuda', enabled=use_amp)

best_val, no_improve, best_epoch = float('inf'), 0, 0
history = []

print('Training Occupancy Network — PointNet + Fourier INR')
print(f'  Epochs: {CFG["epochs"]}  |  Batch: {CFG["batch_size"]}  |  LR: {CFG["lr"]}')
print('-' * 80)
print(f'{"Epoch":>6}  {"Tr Loss":>9}  {"Tr Endo":>9}  {"Tr Epi":>8}  '
      f'{"Val Loss":>9}  {"Val Endo":>9}  {"Val Epi":>8}  {"LR":>9}')
print('-' * 80)

t0 = time.time()
for epoch in range(CFG['epochs']):

    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    tr_loss = tr_endo = tr_epi = 0.0
    n_tr = 0

    for batch in tqdm(tr_loader, desc=f'Ep {epoch:3d}', leave=False):
        contour = batch['contour'].to(DEVICE)          # (B, N, 4)
        mask    = batch['contour_mask'].to(DEVICE)     # (B, N)
        query   = batch['query'].to(DEVICE)            # (B, Q, 3)
        e_occ   = batch['endo_occ'].to(DEVICE)         # (B, Q)
        p_occ   = batch['epi_occ'].to(DEVICE)          # (B, Q)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=use_amp):
            e_logit, p_logit = model(contour, mask, query)
            loss, l_e, l_p   = occupancy_loss(e_logit, p_logit, e_occ, p_occ)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        bs = contour.shape[0]
        tr_loss += loss.item() * bs
        tr_endo += l_e * bs
        tr_epi  += l_p * bs
        n_tr    += bs

    tr_loss /= n_tr; tr_endo /= n_tr; tr_epi /= n_tr

    # ── Validate ─────────────────────────────────────────────────────────────
    model.eval()
    va_loss = va_endo = va_epi = 0.0
    n_va = 0

    with torch.no_grad():
        for batch in val_loader:
            contour = batch['contour'].to(DEVICE)
            mask    = batch['contour_mask'].to(DEVICE)
            query   = batch['query'].to(DEVICE)
            e_occ   = batch['endo_occ'].to(DEVICE)
            p_occ   = batch['epi_occ'].to(DEVICE)

            with torch.amp.autocast('cuda', enabled=use_amp):
                e_logit, p_logit = model(contour, mask, query)
                loss, l_e, l_p   = occupancy_loss(e_logit, p_logit, e_occ, p_occ)

            bs = contour.shape[0]
            va_loss += loss.item() * bs
            va_endo += l_e * bs
            va_epi  += l_p * bs
            n_va    += bs

    va_loss /= n_va; va_endo /= n_va; va_epi /= n_va

    scheduler.step()
    lr_now = scheduler.get_last_lr()[0]

    history.append(dict(epoch=epoch, tr_loss=tr_loss, tr_endo=tr_endo, tr_epi=tr_epi,
                        va_loss=va_loss, va_endo=va_endo, va_epi=va_epi, lr=lr_now))

    # ── Checkpoint ───────────────────────────────────────────────────────────
    if va_loss < best_val:
        best_val, best_epoch, no_improve = va_loss, epoch, 0
        torch.save({
            'epoch': epoch, 'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'val_loss': va_loss, 'cfg': CFG, 'history': history,
        }, CFG['ckpt_path'])
    else:
        no_improve += 1

    if epoch % 10 == 0 or no_improve == 0:
        print(f'{epoch:6d}  {tr_loss:9.4f}  {tr_endo:9.4f}  {tr_epi:8.4f}  '
              f'{va_loss:9.4f}  {va_endo:9.4f}  {va_epi:8.4f}  {lr_now:9.2e}'
              f'  ← best' if no_improve == 0 else '')

    if no_improve >= CFG['patience']:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nBest val_loss={best_val:.4f} at epoch {best_epoch} | '
      f'total {time.time()-t0:.0f}s')


## 9. Inference — Marching Cubes

In [ ]:
def predict_mesh(model, contour_xyz, tissue_labels, cfg,
                  grid_res=None, iso_thresh=None, batch_query=65536):
    """
    Run the occupancy network on a dense voxel grid and extract
    endo + epi meshes via Marching Cubes.

    Parameters
    ----------
    model         : trained OccupancyNetwork
    contour_xyz   : (N, 3) SAX contour positions (already normalised)
    tissue_labels : (N,)   tissue labels (0=endo, 1=epi)
    cfg           : CFG dict
    grid_res      : voxel grid resolution  (default CFG['grid_res'])
    iso_thresh    : iso-surface threshold  (default CFG['iso_thresh'])
    batch_query   : number of query points per GPU call (memory control)

    Returns
    -------
    endo_mesh : trimesh.Trimesh  (endocardial surface)
    epi_mesh  : trimesh.Trimesh  (epicardial surface)
    occ_endo  : (R, R, R) occupancy volume for visualisation
    occ_epi   : (R, R, R) occupancy volume
    """
    if grid_res   is None: grid_res   = cfg['grid_res']
    if iso_thresh is None: iso_thresh = cfg['iso_thresh']

    model.eval()

    # ── Build encoder input ──────────────────────────────────────────────────
    cont = np.column_stack([contour_xyz, tissue_labels]).astype(np.float32)
    cont_t = torch.from_numpy(cont).unsqueeze(0).to(DEVICE)   # (1, N, 4)
    mask_t = torch.ones(1, len(cont), dtype=torch.bool).to(DEVICE)

    with torch.no_grad():
        z = model.encode(cont_t, mask_t)                       # (1, latent_dim)

    # ── Build query grid ─────────────────────────────────────────────────────
    # Grid spans the contour bounding box + small padding
    lo = contour_xyz.min(0) - 0.15
    hi = contour_xyz.max(0) + 0.15
    xs = np.linspace(lo[0], hi[0], grid_res)
    ys = np.linspace(lo[1], hi[1], grid_res)
    zs = np.linspace(lo[2], hi[2], grid_res)
    gx, gy, gz = np.meshgrid(xs, ys, zs, indexing='ij')
    grid_pts   = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=-1).astype(np.float32)

    # ── Query in batches ─────────────────────────────────────────────────────
    endo_logits = np.zeros(len(grid_pts), np.float32)
    epi_logits  = np.zeros(len(grid_pts), np.float32)

    for start in range(0, len(grid_pts), batch_query):
        pts = torch.from_numpy(grid_pts[start:start+batch_query]).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            el, pl = model.decode(z, pts)
        endo_logits[start:start+batch_query] = el[0].cpu().numpy()
        epi_logits[ start:start+batch_query] = pl[0].cpu().numpy()

    # ── Reshape to 3D volumes ─────────────────────────────────────────────────
    occ_endo = torch.sigmoid(torch.from_numpy(endo_logits)).numpy().reshape(grid_res, grid_res, grid_res)
    occ_epi  = torch.sigmoid(torch.from_numpy(epi_logits)).numpy().reshape(grid_res, grid_res, grid_res)

    voxel_size = (hi - lo) / (grid_res - 1)

    def volume_to_mesh(occ_vol):
        if occ_vol.max() < iso_thresh:
            return trimesh.Trimesh()
        try:
            verts_g, faces_g, _, _ = marching_cubes(
                occ_vol, level=iso_thresh,
                spacing=voxel_size)
            # Shift from grid indices to world coords
            verts_w = verts_g + lo
            return trimesh.Trimesh(vertices=verts_w, faces=faces_g, process=True)
        except Exception as e:
            print(f'  Marching cubes failed: {e}')
            return trimesh.Trimesh()

    endo_mesh = volume_to_mesh(occ_endo)
    epi_mesh  = volume_to_mesh(occ_epi)

    return endo_mesh, epi_mesh, occ_endo, occ_epi


# ── Load best checkpoint ──────────────────────────────────────────────────────
ckpt = torch.load(CFG['ckpt_path'], map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded checkpoint — epoch {ckpt["epoch"]}  val_loss={ckpt["val_loss"]:.4f}')


## 10. Wall Thickness Computation

In [ ]:
def compute_wall_thickness(endo_mesh, epi_mesh, n_samples=2000):
    """
    Estimate wall thickness by shooting rays from endo surface outward
    (along normals) and finding the intersection with the epi surface.

    Returns
    -------
    thickness_mm : (n_valid,)  per-sample wall thickness in mm (normalised coords)
    sample_pts   : (n_valid, 3) positions where thickness was measured
    """
    if len(endo_mesh.vertices) == 0 or len(epi_mesh.vertices) == 0:
        return np.array([]), np.array([]).reshape(0, 3)

    # Sample points on endo surface
    pts, face_idx = trimesh.sample.sample_surface(endo_mesh, n_samples)
    normals       = endo_mesh.face_normals[face_idx]

    # Cast rays outward from endo → epi
    ray_origins    = pts + normals * 0.01      # tiny offset to avoid self-hit
    locations, idx_ray, _ = epi_mesh.ray.intersects_location(
        ray_origins=ray_origins,
        ray_directions=normals,
        multiple_hits=False,
    )

    if len(idx_ray) == 0:
        return np.array([]), np.array([]).reshape(0, 3)

    origin_valid  = ray_origins[idx_ray]
    thickness_mm  = np.linalg.norm(locations - origin_valid, axis=1)

    return thickness_mm, origin_valid


## 11. Visualisation — 3D Mesh + Occupancy Slices

In [ ]:
def visualise_prediction(sample_idx=0, n_vis_slices=3):
    """
    Full inference + visualisation pipeline for one test sample.
    Shows:
      (a) Input SAX contours (endo=blue, epi=red)
      (b) Predicted 3D endo mesh
      (c) Predicted 3D epi mesh
      (d) Occupancy cross-section (XY slice through centre)
      (e) Wall thickness map
    """
    m    = te_meta[sample_idx]
    idx  = m.get('saved_idx', sample_idx)
    pts  = load_vtk_points(f'{CFG["mesh_dir"]}/mesh_{idx:04d}.vtk')

    # Extract and normalise contours
    xyz, tissue = extract_sax_contours(
        pts, CFG['n_slices'], CFG['slice_thick'],
        CFG['slice_space'], CFG['pts_per_ring'])

    centroid = xyz[:, :2].mean(0)
    scale    = np.linalg.norm(xyz[:, :2] - centroid, axis=1).mean()
    scale    = max(scale, 1e-3)
    xyz_n    = xyz.copy()
    xyz_n[:, :2] -= centroid
    xyz_n[:, 2]  -= xyz[:, 2].mean()
    xyz_n        /= scale

    # ── Inference ────────────────────────────────────────────────────────────
    endo_mesh, epi_mesh, occ_endo, occ_epi = predict_mesh(
        model, xyz_n, tissue, CFG, grid_res=CFG['grid_res'])

    thickness, thick_pts = compute_wall_thickness(endo_mesh, epi_mesh)

    # ── Ground truth mesh for comparison ────────────────────────────────────
    gt_ev, gt_ef = load_vtk_mesh(f'{CFG["mesh_dir"]}/mesh_{idx:04d}.vtk')
    if len(gt_ef) == 0:
        gt_ev, gt_ef = ENDO_VERTS.copy(), ENDO_FACES.copy()
    gt_ev_n    = gt_ev.copy()
    gt_ev_n[:, :2] -= centroid
    gt_ev_n[:, 2]  -= xyz[:, 2].mean()
    gt_ev_n        /= scale

    # ── Figure ────────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(22, 9))
    gs  = plt.GridSpec(2, 5, figure=fig, hspace=0.35, wspace=0.25)

    # (a) Input SAX contours (3D scatter)
    ax0 = fig.add_subplot(gs[:, 0], projection='3d')
    ax0.set_title('Input SAX Contours\n(blue=endo, red=epi)', fontsize=10, fontweight='bold')
    endo_m = tissue == 0
    epi_m  = tissue == 1
    ax0.scatter(xyz_n[endo_m, 0], xyz_n[endo_m, 1], xyz_n[endo_m, 2],
                c='#3498db', s=8, alpha=0.8, label='Endo')
    ax0.scatter(xyz_n[epi_m,  0], xyz_n[epi_m,  1], xyz_n[epi_m,  2],
                c='#e74c3c', s=8, alpha=0.8, label='Epi')
    ax0.legend(fontsize=7); ax0.set_axis_off()
    ax0.view_init(elev=15, azim=30)

    # (b) Predicted endo mesh
    ax1 = fig.add_subplot(gs[:, 1], projection='3d')
    ax1.set_title(f'Predicted Endo Mesh\n({len(endo_mesh.vertices)} verts)', fontsize=10, fontweight='bold')
    if len(endo_mesh.vertices) > 0:
        ax1.add_collection3d(Poly3DCollection(
            endo_mesh.vertices[endo_mesh.faces[:min(6000, len(endo_mesh.faces))]],
            alpha=0.55, facecolor='#3498db', edgecolor='none'))
        _set_ax_lim(ax1, endo_mesh.vertices)
    ax1.set_axis_off(); ax1.view_init(elev=15, azim=30)

    # (c) Predicted epi mesh
    ax2 = fig.add_subplot(gs[:, 2], projection='3d')
    ax2.set_title(f'Predicted Epi Mesh\n({len(epi_mesh.vertices)} verts)', fontsize=10, fontweight='bold')
    if len(epi_mesh.vertices) > 0:
        ax2.add_collection3d(Poly3DCollection(
            epi_mesh.vertices[epi_mesh.faces[:min(6000, len(epi_mesh.faces))]],
            alpha=0.45, facecolor='#e74c3c', edgecolor='none'))
        _set_ax_lim(ax2, epi_mesh.vertices)
    ax2.set_axis_off(); ax2.view_init(elev=15, azim=30)

    # (d) Both meshes overlaid + GT endo
    ax3 = fig.add_subplot(gs[:, 3], projection='3d')
    ax3.set_title('Endo + Epi Overlay\n+ GT endo (green)', fontsize=10, fontweight='bold')
    if len(endo_mesh.vertices) > 0:
        ax3.add_collection3d(Poly3DCollection(
            endo_mesh.vertices[endo_mesh.faces[:4000]],
            alpha=0.35, facecolor='#3498db', edgecolor='none'))
    if len(epi_mesh.vertices) > 0:
        ax3.add_collection3d(Poly3DCollection(
            epi_mesh.vertices[epi_mesh.faces[:4000]],
            alpha=0.25, facecolor='#e74c3c', edgecolor='none'))
    if len(gt_ef) > 0:
        ax3.add_collection3d(Poly3DCollection(
            gt_ev_n[gt_ef[:4000]],
            alpha=0.40, facecolor='#27ae60', edgecolor='none'))
    if len(endo_mesh.vertices) > 0:
        _set_ax_lim(ax3, np.vstack([endo_mesh.vertices, epi_mesh.vertices]) if len(epi_mesh.vertices) > 0 else endo_mesh.vertices)
    ax3.set_axis_off(); ax3.view_init(elev=15, azim=30)

    # (e) Occupancy cross-section + wall thickness histogram
    ax4a = fig.add_subplot(gs[0, 4])
    mid_z = occ_endo.shape[2] // 2
    im = ax4a.imshow(
        np.concatenate([occ_endo[:, :, mid_z], occ_epi[:, :, mid_z]], axis=1),
        cmap='RdBu_r', vmin=0, vmax=1, origin='lower', aspect='auto')
    ax4a.set_title('Occupancy XY slice\n(left=endo, right=epi)', fontsize=9, fontweight='bold')
    plt.colorbar(im, ax=ax4a, fraction=0.046)

    ax4b = fig.add_subplot(gs[1, 4])
    if len(thickness) > 0:
        ax4b.hist(thickness * scale, bins=30, color='#9b59b6', alpha=0.8, edgecolor='white', lw=0.4)
        ax4b.axvline(np.median(thickness * scale), color='#e74c3c', lw=2,
                     label=f'Median = {np.median(thickness)*scale:.1f} mm')
        ax4b.set_xlabel('Wall Thickness (mm)')
        ax4b.set_ylabel('Count')
        ax4b.set_title('Wall Thickness Distribution', fontsize=9, fontweight='bold')
        ax4b.legend(fontsize=8)
        ax4b.grid(alpha=0.3)
    else:
        ax4b.text(0.5, 0.5, 'No thickness data', ha='center', va='center', transform=ax4b.transAxes)

    fig.suptitle(f'LV Reconstruction — INR Occupancy Network (Test Sample {sample_idx})',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.savefig(f'{CFG["output_dir"]}/inr_sample_{sample_idx}.png',
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

    # ── Print stats ───────────────────────────────────────────────────────────
    print(f'\nSample {sample_idx} stats:')
    print(f'  Endo mesh : {len(endo_mesh.vertices):,} verts  {len(endo_mesh.faces):,} faces')
    print(f'  Epi  mesh : {len(epi_mesh.vertices):,} verts  {len(epi_mesh.faces):,} faces')
    if len(thickness) > 0:
        print(f'  Wall thickness — mean: {(thickness*scale).mean():.2f} mm  '
              f'median: {np.median(thickness)*scale:.2f} mm  '
              f'std: {(thickness*scale).std():.2f} mm')
    return endo_mesh, epi_mesh, thickness * scale


def _set_ax_lim(ax, pts, pad=0.05):
    c = pts.mean(0); h = (pts.max(0) - pts.min(0)).max() / 2 + pad
    ax.set_xlim(c[0]-h, c[0]+h)
    ax.set_ylim(c[1]-h, c[1]+h)
    ax.set_zlim(c[2]-h, c[2]+h)


# ── Run visualisation ─────────────────────────────────────────────────────────
for i in range(min(3, len(te_meta))):
    endo_m, epi_m, wt = visualise_prediction(sample_idx=i)


## 12. Quantitative Evaluation

In [ ]:
def chamfer_distance(pts_a, pts_b, n_sample=1000):
    """One-directional mean Chamfer distance between two point clouds."""
    from sklearn.neighbors import NearestNeighbors
    if len(pts_a) == 0 or len(pts_b) == 0:
        return float('nan')
    # Subsample for speed
    rng = np.random.default_rng(0)
    a = pts_a[rng.choice(len(pts_a), min(n_sample, len(pts_a)), replace=False)]
    b = pts_b[rng.choice(len(pts_b), min(n_sample, len(pts_b)), replace=False)]
    nn = NearestNeighbors(n_neighbors=1, algorithm='kd_tree').fit(b)
    d, _  = nn.kneighbors(a)
    return float(d.mean())


def evaluate_dataset(loader, n_samples=20):
    """
    Evaluate on n_samples from a loader.
    Metrics:
      - Endo Chamfer distance to GT endo surface
      - Epi  Chamfer distance to GT epi  surface
      - Wall thickness mean / std
    """
    results = []
    count   = 0

    for batch in loader:
        for b in range(batch['contour'].shape[0]):
            if count >= n_samples:
                break

            cont   = batch['contour'][b].numpy()      # (N, 4)
            mask   = batch['contour_mask'][b].numpy()  # (N,)
            scale  = batch['scale'][b].item()
            centroid = batch['centroid'][b].numpy()    # (3,)

            xyz_n  = cont[mask, :3]
            tissue = cont[mask, 3]

            e_mesh, p_mesh, _, _ = predict_mesh(model, xyz_n, tissue, CFG)

            # GT occupancy points from query (those with occ=1)
            qpts  = batch['query'][b].numpy()
            e_occ = batch['endo_occ'][b].numpy()
            p_occ = batch['epi_occ'][b].numpy()
            gt_endo_pts = qpts[e_occ > 0.5]
            gt_epi_pts  = qpts[p_occ > 0.5]

            e_cham = chamfer_distance(
                e_mesh.vertices if len(e_mesh.vertices) > 0 else np.zeros((1,3)),
                gt_endo_pts) * scale if len(gt_endo_pts) > 0 else float('nan')

            p_cham = chamfer_distance(
                p_mesh.vertices if len(p_mesh.vertices) > 0 else np.zeros((1,3)),
                gt_epi_pts) * scale if len(gt_epi_pts) > 0 else float('nan')

            wt, _ = compute_wall_thickness(e_mesh, p_mesh)
            wt_mm = wt * scale

            results.append(dict(
                endo_chamfer = e_cham,
                epi_chamfer  = p_cham,
                wt_mean      = float(wt_mm.mean()) if len(wt_mm) > 0 else float('nan'),
                wt_std       = float(wt_mm.std())  if len(wt_mm) > 0 else float('nan'),
            ))
            count += 1

        if count >= n_samples:
            break

    df = pd.DataFrame(results)
    print('\n── Evaluation Results ─────────────────────────────')
    print(f'  Samples evaluated : {len(df)}')
    print(f'  Endo Chamfer (mm) : {df.endo_chamfer.mean():.3f} ± {df.endo_chamfer.std():.3f}')
    print(f'  Epi  Chamfer (mm) : {df.epi_chamfer.mean():.3f}  ± {df.epi_chamfer.std():.3f}')
    print(f'  Wall thickness    : {df.wt_mean.mean():.2f} ± {df.wt_mean.std():.2f} mm')
    print('─────────────────────────────────────────────────────')
    return df


eval_df = evaluate_dataset(te_loader, n_samples=20)


## 13. Training Curves

In [ ]:
hdf = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
cols = ['#2c3e50', '#e74c3c']

for ax, (tr_col, va_col, title) in zip(axes, [
    ('tr_loss', 'va_loss', 'Total Loss (BCE)'),
    ('tr_endo', 'va_endo', 'Endo Occupancy Loss'),
    ('tr_epi',  'va_epi',  'Epi Occupancy Loss'),
]):
    ax.plot(hdf.epoch, hdf[tr_col], color=cols[0], alpha=0.5, lw=1.5, label='Train')
    ax.plot(hdf.epoch, hdf[va_col], color=cols[1], lw=2.0, label='Val')
    ax.axvline(best_epoch, color='#7f8c8d', ls='--', lw=1.2, alpha=0.7,
               label=f'Best (ep {best_epoch})')
    ax.set_xlabel('Epoch'); ax.set_ylabel(title)
    ax.set_title(title, fontweight='bold')
    ax.legend(framealpha=0.9, edgecolor='none')
    ax.grid(alpha=0.25, ls=':')
    for sp in ['top', 'right']:
        ax.spines[sp].set_visible(False)

fig.suptitle('Training Convergence — INR Occupancy Network', fontsize=13,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{CFG["output_dir"]}/inr_training_curves.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved → {CFG["output_dir"]}/inr_training_curves.png')


## 14. Latent Space — Shape Diversity

In [ ]:
"""
One major advantage of the INR approach is that you can explore the latent space.
Here we:
  1. Encode all test samples → z vectors
  2. Compute PCA on the z vectors
  3. Traverse the first 2 PCA modes → see what shape variations they encode
"""

from sklearn.decomposition import PCA as skPCA

# ── Collect all test latents ──────────────────────────────────────────────────
all_z   = []
model.eval()
with torch.no_grad():
    for batch in te_loader:
        cont = batch['contour'].to(DEVICE)
        mask = batch['contour_mask'].to(DEVICE)
        z    = model.encode(cont, mask)
        all_z.append(z.cpu().numpy())

Z_mat = np.vstack(all_z)   # (N_test, latent_dim)
print(f'Latent matrix: {Z_mat.shape}')

# ── PCA on latent space ───────────────────────────────────────────────────────
pca_z  = skPCA(n_components=2)
Z_2d   = pca_z.fit_transform(Z_mat)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 2D scatter of latent space
ax = axes[0]
sc = ax.scatter(Z_2d[:, 0], Z_2d[:, 1], c=np.arange(len(Z_2d)),
                cmap='plasma', s=30, alpha=0.8, edgecolors='none')
plt.colorbar(sc, ax=ax, label='Sample index')
ax.set_xlabel('Latent PC1'); ax.set_ylabel('Latent PC2')
ax.set_title('Latent Space (2D PCA projection)', fontweight='bold')
ax.grid(alpha=0.3); ax.spines[['top','right']].set_visible(False)

# Variance explained
cumvar = np.cumsum(pca_z.explained_variance_ratio_)
axes[1].bar([1,2], pca_z.explained_variance_ratio_ * 100,
            color=['#3498db', '#e74c3c'], alpha=0.8, edgecolor='white')
axes[1].set_xlabel('PC'); axes[1].set_ylabel('Variance explained (%)')
axes[1].set_title('Latent PCA — Variance Explained', fontweight='bold')
axes[1].grid(alpha=0.3, axis='y')
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(f'{CFG["output_dir"]}/inr_latent_space.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()
print(f'Cumulative variance (PC1+2): {cumvar[1]*100:.1f}%')


## 15. Save Results

In [ ]:
# Save eval results
eval_df.to_csv(f'{CFG["output_dir"]}/inr_eval_results.csv', index=False)
print(f'✅ Eval CSV → {CFG["output_dir"]}/inr_eval_results.csv')

# Save full run summary
summary = dict(
    best_epoch    = best_epoch,
    best_val_loss = best_val,
    n_epochs      = len(history),
    endo_chamfer_mean = float(eval_df.endo_chamfer.mean()),
    epi_chamfer_mean  = float(eval_df.epi_chamfer.mean()),
    wall_thickness_mean = float(eval_df.wt_mean.mean()),
    cfg    = CFG,
    history = history,
)
with open(f'{CFG["output_dir"]}/inr_run_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'✅ Run summary → {CFG["output_dir"]}/inr_run_summary.json')

# Final checkpoint
torch.save({
    'epoch'        : best_epoch,
    'model_state'  : model.state_dict(),
    'val_loss'     : best_val,
    'cfg'          : CFG,
    'eval_results' : eval_df.to_dict(),
    'history'      : history,
}, f'{CFG["output_dir"]}/inr_final.pt')
print(f'✅ Final checkpoint → {CFG["output_dir"]}/inr_final.pt')
print(f'\n   Best val loss : {best_val:.4f}   at epoch {best_epoch}')
print(f'   Endo Chamfer  : {eval_df.endo_chamfer.mean():.3f} mm')
print(f'   Epi Chamfer   : {eval_df.epi_chamfer.mean():.3f} mm')
print(f'   Wall thickness: {eval_df.wt_mean.mean():.2f} ± {eval_df.wt_mean.std():.2f} mm')
